# Chapitre 3 - Cours 5 : Transactionnel ou analytique, et le stockage en colonnes

Ce notebook couvre la seconde moitié du chapitre : la section *Transaction Processing or
Analytics?* (*Data Warehousing*, *The divergence between OLTP databases and data
warehouses*, *Stars and Snowflakes: Schemas for Analytics*), la section *Column-Oriented
Storage* (*Column Compression*, *Memory bandwidth and vectorized processing*, *Sort
Order in Column Storage*, *Several different sort orders*, *Writing to Column-Oriented
Storage*) et la section *Aggregation: Data Cubes and Materialized Views*.

**Plan.**

1. Transactionnel ou analytique ? Les entrepôts de données et l'ETL.
2. Étoiles et flocons : les schémas pour l'analytique.
3. Le stockage orienté colonnes.
4. La compression des colonnes : dictionnaire, bitmaps, codage par plages.
5. Bande passante mémoire et traitement vectorisé.
6. L'ordre de tri dans un stockage en colonnes.
7. Écrire dans un stockage en colonnes.
8. Agrégation : vues matérialisées et cubes de données.

In [1]:
import gc
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import storage as S
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
fact, dim_product, dim_date, dim_store = S.make_sales(500_000, n_extra_cols=12, seed=0)


def meilleur_temps(f, rep=5):
    """Meilleur temps (s) de `rep` exécutions, ramasse-miettes désactivé."""
    best = float("inf")
    for _ in range(rep):
        gc.disable()
        t0 = time.perf_counter()
        f()
        best = min(best, time.perf_counter() - t0)
        gc.enable()
    return best